# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

---
### Signal checks first (bucket tables with n are in the cell below)

| Signal | Behind which FlyRank flag | What I expected | Verdict |
|---|---|---|---|
| Staleness (`days_since_last_update`) | the refresh / stale-page flags | older update → more drops | **MIXED** |
| Earlier-window CTR (`clicks_prev_30d / impressions_prev_30d`) | the CTR-fix logic | low CTR → more drops | **CONFIRMED** |
| *(bonus)* Content age | "old content decays" intuition | older page → more drops | **OPPOSITE** |

- **Staleness — MIXED.** The two big buckets differ by only ~2 points (0.61 vs 0.63). The 181+ bucket
  looks huge (≈95%) but has only ~21 pages, so I don't trust it. Also ~60% of slice pages show an update
  *inside* the last 30 days — that is the label window, so this column partly describes the future
  from the decision moment. **I keep it out of the rule.**
- **Earlier-window CTR — CONFIRMED.** Pages with ≤ 0.1% CTR in days 31–60 dropped ~67% of the time,
  pages above 0.5% about 48–51%. I use CTR alone, not CTR-vs-position, because `avg_position` is a
  90-day mean that overlaps the label window.
- **Content age — OPPOSITE** to the usual intuition: in this slice **younger** pages (90–180 days)
  drop most often (~71%) and 365+ pages least (~45%). Young pages are still settling in search.
  A clearly-explained negative — the rule leans on youth, not on age.

### My rule in plain words
> *A page goes to the top of the review queue when it is still young, got few or no clicks from the
> impressions it already had, and had real demand in the earlier window.*

`baseline_score = 0.4 × youth + 0.4 × weak_ctr + 0.2 × demand` — each part is a percentile rank
(0–1) of an earlier-window or static column, weights chosen by hand, nothing fitted.

### Reason codes → action labels (one reason code per page, first match wins)
| Reason code | Condition | Action label |
|---|---|---|
| `ZERO_CLICK_YOUNG_PAGE` | age ≤ 180 days and 0 clicks in days 31–60 | `rewrite_title_and_meta` |
| `WEAK_CTR_WITH_DEMAND` | earlier CTR < 0.5% and ≥ 1,000 earlier impressions | `refresh_snippet_and_intro` |
| `YOUNG_PAGE_SETTLING` | age ≤ 180 days | `light_refresh_then_monitor` |
| `LOW_RISK` | none of the above | `monitor` |

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)

lane["ctr_prev"] = lane["clicks_prev_30d"] / lane["impressions_prev_30d"] * 100   # x100 percent

def bucket_table(col, bins, labels):
    b = pd.cut(lane[col], bins=bins, labels=labels, include_lowest=True)
    t = lane.groupby(b, observed=True)["demand_drop"].agg(n="size", drop_rate="mean")
    t["drop_rate"] = t["drop_rate"].round(3)
    return t

print(f"base rate (all slice pages): {lane['demand_drop'].mean():.3f}  n={len(lane):,}\n")
print("SIGNAL 1 - staleness (days since last update)  [refresh-flag signal]")
display(bucket_table("days_since_last_update", [0, 30, 90, 180, 400], ["0-30", "31-90", "91-180", "181+"]))
print(f"share of pages updated inside the last 30 days (label window): "
      f"{(lane['days_since_last_update'] <= 30).mean():.1%}")
print("\nSIGNAL 2 - earlier-window CTR (%)  [CTR-fix signal]")
display(bucket_table("ctr_prev", [-0.01, 0.1, 0.5, 1, 2, 100], ["<=0.1", "0.1-0.5", "0.5-1", "1-2", ">2"]))
print("\nBONUS - content age (days)")
display(bucket_table("content_age_days", [89, 180, 365, 5000], ["90-180", "181-365", "365+"]))
verdicts = {"staleness": "MIXED", "earlier_window_ctr": "CONFIRMED", "content_age": "OPPOSITE"}
print(verdicts)

rows x cols: (30000, 44)
base rate (all slice pages): 0.616  n=18,010

SIGNAL 1 - staleness (days since last update)  [refresh-flag signal]


,n,drop_rate
days_since_last_update,,
0-30,10900,0.608
31-90,85,0.588
91-180,7004,0.628
181+,21,0.952


share of pages updated inside the last 30 days (label window): 60.5%

SIGNAL 2 - earlier-window CTR (%)  [CTR-fix signal]


,n,drop_rate
ctr_prev,,
<=0.1,8642,0.673
0.1-0.5,6399,0.591
0.5-1,2213,0.509
1-2,664,0.483
>2,92,0.489



BONUS - content age (days)


,n,drop_rate
content_age_days,,
90-180,7394,0.713
181-365,6404,0.613
365+,4212,0.450


{'staleness': 'MIXED', 'earlier_window_ctr': 'CONFIRMED', 'content_age': 'OPPOSITE'}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

---
The cell below codes the rule exactly as written above, ranks all 18,010 slice pages, and writes
the queue to `work/outputs/baseline_action_score.csv` (gitignored on purpose — the notebook
regenerates it). The run's receipts go to `work/outputs/w04_baseline_metrics.json` (committed).

Quick evaluation, always next to the base rate: precision@10/50/100 on the whole slice, and —
closer to how an editor works — precision@10 **inside each client's own queue**.

In [2]:
import json
OUT = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
OUT.mkdir(parents=True, exist_ok=True)

pct = lambda s: s.rank(pct=True)
lane["youth"] = 1 - pct(lane["content_age_days"])
lane["weak_ctr"] = 1 - pct(lane["ctr_prev"])
lane["demand"] = pct(np.log1p(lane["impressions_prev_30d"]))
lane["baseline_score"] = (0.4 * lane["youth"] + 0.4 * lane["weak_ctr"] + 0.2 * lane["demand"]).round(4)

def reason_code(r):
    if r["content_age_days"] <= 180 and r["clicks_prev_30d"] == 0:
        return "ZERO_CLICK_YOUNG_PAGE"
    if r["ctr_prev"] < 0.5 and r["impressions_prev_30d"] >= 1000:
        return "WEAK_CTR_WITH_DEMAND"
    if r["content_age_days"] <= 180:
        return "YOUNG_PAGE_SETTLING"
    return "LOW_RISK"

ACTION = {"ZERO_CLICK_YOUNG_PAGE": "rewrite_title_and_meta",
          "WEAK_CTR_WITH_DEMAND": "refresh_snippet_and_intro",
          "YOUNG_PAGE_SETTLING": "light_refresh_then_monitor",
          "LOW_RISK": "monitor"}
lane["reason_code"] = lane.apply(reason_code, axis=1)
lane["action_label"] = lane["reason_code"].map(ACTION)

queue = lane.sort_values("baseline_score", ascending=False, kind="stable").reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
cols = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action_label",
        "content_age_days", "impressions_prev_30d", "clicks_prev_30d", "ctr_prev",
        "content_type", "main_intent", "word_count", "search_volume", "demand_drop"]
queue[cols].to_csv(OUT / "baseline_action_score.csv", index=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

y, sc = lane["demand_drop"].values, lane["baseline_score"].values
base = float(y.mean())
metrics = {"slice_rows": int(len(lane)), "clients": int(lane["client_id"].nunique()),
           "base_rate": round(base, 4)}
for k in (10, 50, 100):
    metrics[f"precision_at_{k}"] = round(precision_at_k(sc, y, k), 4)
per_client = [precision_at_k(g["baseline_score"].values, g["demand_drop"].values, 10)
              for _, g in lane.groupby("client_id") if len(g) >= 50]
per_client_base = [g["demand_drop"].mean() for _, g in lane.groupby("client_id") if len(g) >= 50]
metrics["per_client_precision_at_10_median"] = round(float(np.median(per_client)), 4)
metrics["per_client_base_rate_median"] = round(float(np.median(per_client_base)), 4)
metrics["clients_with_50plus_pages"] = len(per_client)
metrics["reason_code_counts"] = queue["reason_code"].value_counts().to_dict()
metrics["score_formula"] = {"youth": 0.4, "weak_ctr": 0.4, "demand": 0.2}
(OUT / "w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print("wrote", OUT / "baseline_action_score.csv", "rows:", len(queue))
print(json.dumps(metrics, indent=2))

wrote ../outputs/baseline_action_score.csv rows: 18010
{
  "slice_rows": 18010,
  "clients": 30,
  "base_rate": 0.6158,
  "precision_at_10": 0.8,
  "precision_at_50": 0.82,
  "precision_at_100": 0.82,
  "per_client_precision_at_10_median": 0.8,
  "per_client_base_rate_median": 0.6469,
  "clients_with_50plus_pages": 18,
  "reason_code_counts": {
    "LOW_RISK": 6657,
    "WEAK_CTR_WITH_DEMAND": 6233,
    "ZERO_CLICK_YOUNG_PAGE": 2760,
    "YOUNG_PAGE_SETTLING": 2360
  },
  "score_formula": {
    "youth": 0.4,
    "weak_ctr": 0.4,
    "demand": 0.2
  }
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

---
(The card asks for the top ten; the skeleton hint says 20 — I review ten here.)

All ten top pages share one shape: **about 3 months old, 1,700–7,100 impressions in days 31–60,
and zero clicks** → reason code `ZERO_CLICK_YOUNG_PAGE`, action `rewrite_title_and_meta`.
Observed outcome: **8 of 10 had a demand drop** (precision@10 = 0.80 vs base rate 0.62).
Ids are the last 6 characters of the pseudonym; "client" is a number local to this table.

| # | page | action | why it's here | what would make it wrong | confidence |
|---|---|---|---|---|---|
| 1 | `c11499` | rewrite_title_and_meta | 97 days old, ~7.1k earlier impressions, 0 clicks; commercial intent with a real keyword (vol 40) | impressions come from deep positions / off-topic queries, so a new title won't earn clicks | medium |
| 2 | `cd9ba7` | rewrite_title_and_meta | 95 days, ~3.6k impressions, 0 clicks | keyword volume is 0 — demand may be broad noise; **it did not drop** (weak pick) | low |
| 3 | `61f858` | rewrite_title_and_meta | 90 days, ~2.2k impressions, 0 clicks | page is still being tested by search and would settle by itself without any edit | medium |
| 4 | `66aacf` | rewrite_title_and_meta | 98 days, ~3.9k impressions, 0 clicks | another page of the same client targets the same topic (cannibalization) — fix is merge, not rewrite | medium |
| 5 | `e3c008` | rewrite_title_and_meta | 104 days, ~5.1k impressions, 0 clicks, long article (3.5k words) | **did not drop**; same client as #2 and #9 — the rule may over-rank one client's template | low |
| 6 | `109516` | rewrite_title_and_meta | 90 days, ~1.7k impressions, 0 clicks, transactional | transactional queries may be won by shops/ads, so CTR stays near 0 whatever the title says | medium |
| 7 | `63e130` | rewrite_title_and_meta | 90 days, ~1.7k impressions, 0 clicks | the drop is seasonal (topic demand fell), not a page problem | medium |
| 8 | `20e33b` | rewrite_title_and_meta | 96 days, ~2.7k impressions, 0 clicks, best keyword demand in the top 10 (vol 210) | if position is page 3+ the problem is ranking, not the snippet — needs a content refresh instead | medium-high |
| 9 | `20808e` | rewrite_title_and_meta | 98 days, ~3.4k impressions, 0 clicks | same client as #2/#5 — maybe a tracking or site-wide issue, which no page edit fixes | low-medium |
| 10 | `49f25d` | rewrite_title_and_meta | 95 days, ~2.1k impressions, 0 clicks | keyword volume only 10; the impressions may be one-off and not worth editor time | medium |

In [3]:
top10 = queue[cols].head(10).copy()
top10["content_id"] = top10["content_id"].str[-6:]          # short pseudonym, enough to point at a row
top10["client"] = top10["client_id"].astype("category").cat.codes  # client A/B/C.. as a number, no id printed
top10.drop(columns="client_id").round({"ctr_prev": 2})

,rank,content_id,baseline_score,reason_code,action_label,content_age_days,impressions_prev_30d,clicks_prev_30d,ctr_prev,content_type,main_intent,word_count,search_volume,demand_drop,client
0,1,c11499,0.8786,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,97,7078,0,0.0,keyword article,commercial,2952.0,40.0,1,5
1,2,cd9ba7,0.8715,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,95,3572,0,0.0,keyword article,informational,3139.0,0.0,0,2
2,3,61f858,0.8651,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,90,2155,0,0.0,keyword article,informational,2735.0,0.0,1,3
3,4,66aacf,0.8581,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,98,3942,0,0.0,keyword article,informational,2950.0,0.0,1,0
4,5,e3c008,0.8561,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,104,5120,0,0.0,keyword article,informational,3474.0,0.0,0,2
5,6,109516,0.8560,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,90,1715,0,0.0,keyword article,transactional,2514.0,10.0,1,4
6,7,63e130,0.8547,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,90,1665,0,0.0,keyword article,informational,2708.0,0.0,1,3
7,8,20e33b,0.8545,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,96,2665,0,0.0,keyword article,commercial,2803.0,210.0,1,1
8,9,20808e,0.8530,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,98,3350,0,0.0,keyword article,informational,3736.0,0.0,1,2
9,10,49f25d,0.8525,ZERO_CLICK_YOUNG_PAGE,rewrite_title_and_meta,95,2125,0,0.0,keyword article,informational,2786.0,10.0,1,5


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

---
**Weak picks I see:**
- **#2 and #5 did not drop** although they rank top-5. Both belong to the same client as #9 — three of
  the top ten come from one client. A rule that ranks globally can get pulled toward one client's
  pattern (e.g. a template that never earns clicks). Per-client queues (median precision@10 = 0.80
  inside each client's own list, cell above) are the safer way to hand this to editors.
- **Volume blind spot.** In the top 50, most pages have keyword volume ≤ 10 (cell below). "Zero clicks
  on thousands of impressions" can simply mean broad, low-intent impressions — rewriting the title will
  not help those. Next step: add a minimum keyword-volume or position-window check.
- **One reason code dominates.** 48 of the top 50 are `ZERO_CLICK_YOUNG_PAGE`, so the queue's top is
  one idea repeated. That's fine for a baseline, but the editor sees little variety.
- **Action is a guess.** "Rewrite title/meta" is my mapping from a reason code, not a measured fix —
  a drop says *look here*, not *this edit will help*.

**Leakage check (cell below):** the rule reads only `content_age_days`, `clicks_prev_30d`,
`impressions_prev_30d` and their ratio — all earlier-window or static. No `trend_*`, no last-30-day
columns, no 90-day totals/averages (they overlap the label window), no `days_since_last_update`
(updates inside the label window), and no product flags or tiers. As a sanity proof, a score built
from the last-30-day column gets precision@50 = 1.00 — that's what leakage looks like, and why it is
banned. My honest rule's 0.82 is the number I keep, and it is the baseline the week-5 model must beat.

In [4]:
# Leakage check: the rule may only read earlier-window or static columns.
RULE_INPUTS = ["content_age_days", "clicks_prev_30d", "impressions_prev_30d", "ctr_prev"]
FORBIDDEN = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
             "sessions_last_30d", "impressions_90d", "clicks_90d", "ctr", "avg_position",
             "days_since_last_update", "freshness_tier", "impression_tier", "position_tier",
             "is_declining_label"]
assert not set(RULE_INPUTS) & set(FORBIDDEN)
print("rule inputs:", RULE_INPUTS, "-> no label-window, 90-day-overlap, or product-flag columns")

# Proof the check matters: a label-window score is "too good"
leaky = -(lane["impressions_last_30d"] / lane["impressions_prev_30d"])
print(f"precision@50  honest rule: {precision_at_k(sc, y, 50):.2f} | "
      f"leaky last-30d score: {precision_at_k(leaky.values, y, 50):.2f}  <- never use")

# Weak-pick check: how often the top 50 are pages with tiny search volume, and client spread
top50 = queue.head(50)
print(f"top-50: {top50['client_id'].nunique()} clients, "
      f"{(top50['search_volume'].fillna(0) <= 10).mean():.0%} have search_volume <= 10, "
      f"precision {top50['demand_drop'].mean():.2f}")

rule inputs: ['content_age_days', 'clicks_prev_30d', 'impressions_prev_30d', 'ctr_prev'] -> no label-window, 90-day-overlap, or product-flag columns
precision@50  honest rule: 0.82 | leaky last-30d score: 1.00  <- never use
top-50: 11 clients, 78% have search_volume <= 10, precision 0.82


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.